In [1]:
import numpy as np
from array import array
import ROOT
import os

In [2]:
file = ROOT.TFile.Open("stats_Sr_Run0_Ch0-550V_trig2750V.root")

In [3]:
df = ROOT.RDataFrame("Analysis", "stats_Sr_Run0_Ch0-550V_trig2750V.root")

block_lenght = 10000 
number_entries = df.Count().GetValue()
number_blocks= int((number_entries/block_lenght) +1)

In [4]:
Analysis = file.Get("Analysis")

In [6]:
i=0

c1 = ROOT.TCanvas("c1", "canvas", 800, 600)

fit_func = ROOT.TF1("fit_func", "landau",0, 180) 


with open("histo_results.txt", "w") as f_out:
    f_out.write(
        "n_block\tMPV[mV*ns]\tMPV_error[mV*ns]\tsigma[mV*ns]\tChi^2/NDF\tQ[fC]\tQ_error[fC]\n"
    )

if not os.path.exists("Histo_area_new"):
    os.makedirs("Histo_area_new")


while i < number_blocks :
    hist_name = f"h_{i}"
    Analysis.Draw(f"area_new[0]>>{hist_name}(180 0,0,130)", f"event>={i*10000} && event<{(i+1)*10000} && pmax[0]>20 && pmax[0]<240")
    h = ROOT.gDirectory.Get(hist_name)    
   
    if not h or str(type(h)) == "<class 'cppyy.gbl.TObject'>":
        print(f"Errore: impossibile creare l'istogramma {hist_name}. Verifica che 'Area_new' esista.")
        break
        
    h.SetFillColor(ROOT.kBlue-9)
    h.SetLineColor(ROOT.kBlue)
    #h.GetYaxis().SetRangeUser(0, 200);
    h.SetTitle("Distribuzione Area_new;Area_new;Eventi")
    
    fit= h.Fit(fit_func, "RSQ")
    MPV= fit_func.GetParameter(1)
    MPV_error=fit_func.GetParError(1)
    sigma=fit_func.GetParameter(2)
    chi_2= fit_func.GetChisquare()
    NDF= fit_func.GetNDF()
    chi_2_red= chi_2/NDF

    # Area - Current Conversion
    q= float(MPV/4.7)
    q_error=float(MPV_error/4.7)
    
    h.Draw()
    c1.Update()
    c1.Draw()
    c1.SaveAs(f"Histo_area_new/Istogramma_blocco_{i}.png")


    with open("histo_results.txt", "a") as f_out:
            f_out.write(
            f"{i}\t{MPV}\t{MPV_error}\t{sigma}\t{chi_2_red}\t{q}\t{q_error}\n")
    
    i=i+1

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_0.png has been created
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_1.png has been created
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_2.png has been created
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_3.png has been created
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_4.png has been created
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_5.png has been created
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_6.png has been created
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_7.png has been created
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_8.png has been created
Info in <TCanvas::Print>: png file Histo_area_new/Istogramma_blocco_9.png has been created
Info in <TCanvas::Pr

### Q over time

In [7]:
t=[]
Q=[]
Q_error=[]

with open("histo_results.txt", "r") as f:
    lines = f.readlines()
    for line in lines[1:]:
        Values = line.strip().split('\t')
        Q.append(float(Values[5]))
        Q_error.append(float(Values[6]))
        t.append(0.5*float(lines.index(line)))

c2 = ROOT.TCanvas("c2", "canvas", 800, 600)
        
x_Data = array('d', t)

y_Data = array('d', Q)
y_Data_error= array('d', Q_error)


graph_Data = ROOT.TGraphErrors(len(x_Data), x_Data, y_Data, 0 ,y_Data_error)
graph_Data.GetYaxis().SetRangeUser(4, 7);
graph_Data.SetTitle(f"Q(t); t[h];Q[fC]")

graph_Data.SetMarkerStyle(20)      
graph_Data.SetMarkerSize(0.3)
graph_Data.SetMarkerColor(ROOT.kBlack)
ROOT.gStyle.SetEndErrorSize(0)

graph_Data.Draw("APL")

linear_fit = ROOT.TF1("linear_fit", "[0]", 3.49 , 47)
linear_fit.SetParameter(0,5.70)
fit=graph_Data.Fit(linear_fit,"RSQ")
chi_2_linear_fit = linear_fit.GetChisquare()
NDF_linear_fit = fit_func.GetNDF()
chi_2_red_linear_fit  = chi_2_linear_fit/NDF_linear_fit
Q_fit = linear_fit.GetParameter(0) 

c2.Draw()
print(f"Chi^2 ridotto = {chi_2_red_linear_fit}")
print(f"Q = {Q_fit}")



Chi^2 ridotto = 1.6219873237328815
Q = 5.647605538721
